# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Show dataset metadata
md = dataset.metadata
print(f"Dataset: {md.name}\n\nDescription: {md.description}\n\nVersion: {md.version}\n\nLicense: {md.license}")

## 2. Data Overview
Explore available record sets, their IDs, fields and columns.

In [ ]:
# List available record sets by @id and name
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"  - @id: {rs['@id']}")
    print(f"    name: {rs.get('name', '(no name)')}")
    print(f"    description: {rs.get('description', '')}")
    if 'field' in rs:
        print(f"    Fields:")
        for field in rs['field']:
            print(f"      - @id: {field.get('@id', '(no id)')}, name: {field.get('name','(no name)')}")
    if 'column' in rs:
        print(f"    Columns:")
        for col in rs['column']:
            print(f"      - @id: {col.get('@id', '(no id)')}, name: {col.get('name','(no name)')}")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. All references use the record set and field `@id`s from the previous overview.

In [ ]:
# Dynamically collect all record set @id values
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    try:
        # records yields dicts where field-keys are by field @id
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows for record set: {record_set_id}")
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")

# Pick the primary record set (first one)
if len(record_set_ids):
    main_rs_id = record_set_ids[0]
    print(f"\nSample columns in DataFrame for record set {main_rs_id}:")
    print(list(dataframes[main_rs_id].columns))
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filter, normalize numeric fields, and group by a categorical field.

> 🔍 _Remember: all columns are referenced by their `@id`, not display name._

In [ ]:
# EDA on the first record set using field @id
df = dataframes[main_rs_id]

# Print all available columns (field @ids)
print("Columns in main DataFrame:")
print(list(df.columns))

# Example: select a numeric field by @id (edit this to match a real numeric @id)
# We'll try to pick one that looks like abundance, MW, or count
import re
numeric_candidate_ids = [col for col in df.columns if re.search(r'abund|count|mw|intensity|quantity|coverage', col, re.I)]
if numeric_candidate_ids:
    numeric_field_id = numeric_candidate_ids[0]
else:
    numeric_field_id = df.select_dtypes('number').columns[0] if len(df.select_dtypes('number').columns) else df.columns[0]
print(f"Using numeric field @id for analysis: {numeric_field_id}")

# Ensure numeric type
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].mean() # Example threshold: mean value
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} records")
display(filtered_df.head())

# Normalize field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} (z-score) added as new column:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by another field, e.g. one called group/condition/sample/type
group_candidates = [col for col in df.columns if re.search(r'condition|group|sample|type|category', col, re.I)]
group_field = group_candidates[0] if group_candidates else None
if group_field:
    print(f"Grouping by field @id: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    display(grouped_df.head())
else:
    print("No suitable group/categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field or its normalized form.

Here, we show a histogram of the chosen numeric field, colored by an available group (if any).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
if group_field:
    sns.histplot(filtered_df, x=numeric_field_id, hue=group_field, kde=True, bins=30)
else:
    sns.histplot(filtered_df, x=numeric_field_id, kde=True, bins=30)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

## 6. Conclusion

- We loaded a mass spectrometry dataset described with the Croissant schema using the `mlcroissant` library.
- Fields, record sets, and columns were referenced by their `@id`s per FAIR practice.
- Data processing, filtering, normalization, and simple grouping were demonstrated.
- With knowledge of the `@id`s, you can adapt this notebook to other Croissant-based datasets easily.

For more advanced analyses, see [mlcroissant documentation](https://github.com/mlcommons/croissant) and field-level schema details.